In [14]:
import torch
import torch.nn as nn


class Branch(nn.Module):

    def __init__(self, in_ch):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv1d(in_ch, in_ch//2, 2),
            nn.ReLU()
        )

        self.channel_mix = nn.Sequential(
            nn.Conv1d(in_ch//2, in_ch//4, 1),
            nn.ReLU()
        )

    def forward(self, x, B, T, K):

        # x: (B, T, C, K)

        # treat each joint as independent sequence
        x = x.permute(0,3,2,1)        # (B, K, C, T)
        x = x.reshape(B*K, x.shape[2], T)   # (B*K, C, T)

        # temporal motion detection
        x = self.temporal(x)          # (B*K, C/2, T-1)

        # feature interaction
        x = self.channel_mix(x)       # (B*K, C/4, T-1)

        # pool time
        x = x.mean(-1)                # (B*K, C/4)

        # restore batch
        x = x.reshape(B, K, -1)

        # combine joints
        x = x.mean(1)                 # (B, C/4)

        return x

class TemporalClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.b1 = Branch(512)
        self.b2 = Branch(256)
        self.b3 = Branch(128)

        self.cls = nn.Sequential(
            nn.Linear(224, 2)
        )

    def forward(self, x):

        B, T, C, K = x.shape

        x1 = x[:, :, :512, :]
        x2 = x[:, :, 512:768, :]
        x3 = x[:, :, 768:896, :]

        f1 = self.b1(x1, B, T, K)
        f2 = self.b2(x2, B, T, K)
        f3 = self.b3(x3, B, T, K)

        feat = torch.cat([f1, f2, f3], dim=1)

        return self.cls(feat)
    

In [15]:
from torch.cuda import device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = TemporalClassifier().to(device)
model.load_state_dict(torch.load("fight_detector.pth", map_location=device))
model.eval()

Using device: cpu


TemporalClassifier(
  (b1): Branch(
    (temporal): Sequential(
      (0): Conv1d(512, 256, kernel_size=(2,), stride=(1,))
      (1): ReLU()
    )
    (channel_mix): Sequential(
      (0): Conv1d(256, 128, kernel_size=(1,), stride=(1,))
      (1): ReLU()
    )
  )
  (b2): Branch(
    (temporal): Sequential(
      (0): Conv1d(256, 128, kernel_size=(2,), stride=(1,))
      (1): ReLU()
    )
    (channel_mix): Sequential(
      (0): Conv1d(128, 64, kernel_size=(1,), stride=(1,))
      (1): ReLU()
    )
  )
  (b3): Branch(
    (temporal): Sequential(
      (0): Conv1d(128, 64, kernel_size=(2,), stride=(1,))
      (1): ReLU()
    )
    (channel_mix): Sequential(
      (0): Conv1d(64, 32, kernel_size=(1,), stride=(1,))
      (1): ReLU()
    )
  )
  (cls): Sequential(
    (0): Linear(in_features=224, out_features=2, bias=True)
  )
)

In [16]:
import torch
import torch.onnx

model.eval()

dummy_input = torch.randn(1, 8, 896, 15).to(device)  # (B, T, C, 15)

onnx_path = "temporal_classifier.onnx"

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=["input"],
        output_names=["output"],
        opset_version=11
    )

W0310 19:14:30.982000 5620 torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `TemporalClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TemporalClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Program Files\Python312\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python3

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


OSError: [Errno 22] Invalid argument: 'temporal_classifier.onnx.data'

In [ ]:
import torch
import numpy as np
import onnxruntime as ort

# ---- 1. Prepare model ----
model.eval()

dummy_input = torch.randn(1, 8, 896, 15).to(device)  # (B, T, C, 15)

with torch.no_grad():
    pytorch_output = model(dummy_input).cpu().numpy()

# ---- 2. Load ONNX model ----
ort_session = ort.InferenceSession("temporal_classifier.onnx")

# ONNX expects numpy input
onnx_input = dummy_input.cpu().numpy()

# ---- 3. Run ONNX inference ----
ort_inputs = {"input": onnx_input}
onnx_output = ort_session.run(None, ort_inputs)[0]

# ---- 4. Compare ----
max_diff = np.abs(pytorch_output - onnx_output).max()

print("Max difference:", max_diff)

Max difference: 2.9802322e-08


In [ ]:
import time
import numpy as np
import onnxruntime as ort
import torch

# Prepare session
ort_session = ort.InferenceSession("temporal_classifier.onnx")

# Prepare input
dummy_input = torch.randn(1, 8, 896, 15).numpy().astype(np.float32)
ort_inputs = {"input": dummy_input}

# ---- Warmup (important) ----
for _ in range(20):
    ort_session.run(None, ort_inputs)

# ---- Timing ----
runs = 500
start = time.time()

for _ in range(runs):
    ort_session.run(None, ort_inputs)

end = time.time()

avg_time = (end - start) / runs

print("Average inference time (seconds):", avg_time)
print("Average inference time (ms):", avg_time * 1000)
print("Equivalent FPS:", 1 / avg_time)

Average inference time (seconds): 0.0017930550575256349
Average inference time (ms): 1.7930550575256348
Equivalent FPS: 557.7073586239854
